In [4]:
%pip install lxml
%pip install html5lib


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import requests, time, pandas as pd
from io import StringIO

HEADERS = {"User-Agent": "UH-student-project/1.0 (mgonzalez131@uh.edu)"}   # TODO: your preferred contact or a project URL

for name in ["Texas", "California", "Chicago"]:
    url = "https://en.wikipedia.org/wiki/List_of_Michelin-starred_restaurants_in_" + name
    resp = requests.get(url, headers=HEADERS, timeout=15)
    print("====", name, resp.status_code)
    tables = pd.read_html(StringIO(resp.text))
    for i, t in enumerate(tables):
        print(i, t.shape, list(t.columns)[:12])
    time.sleep(1)

==== Texas 200
0 (3, 2) [0, 1]
1 (22, 6) ['Name', 'Cuisine', 'Location', '2024', '2025', '2026']
2 (15, 2) ['vteLists of restaurants', 'vteLists of restaurants.1']
3 (2, 2) [0, 1]
4 (1, 2) [0, 1]
5 (3, 2) [0, 1]
6 (20, 2) ['vteMichelin', 'vteMichelin.1']
7 (12, 2) [0, 1]
8 (6, 2) [0, 1]
9 (3, 2) [0, 1]
==== California 200
0 (5, 2) [0, 1]
1 (93, 11) ['Name', 'Cuisine', 'Location', '2019', '2020 No Guide', '2021', '2022', '2023', '2024', '2025', '2026']
2 (5, 2) [0, 1]
3 (64, 11) ['Name', 'Cuisine', 'Location', '2019', '2020 No Guide', '2021', '2022', '2023', '2024', '2025', '2026']
4 (4, 2) [0, 1]
5 (24, 6) ['Name', 'Cuisine', 'Location', '2008', '2009', '2010–2018 No Guide']
6 (5, 2) [0, 1]
7 (96, 16) ['Name', 'Cuisine', 'Location', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015']
8 (15, 2) ['vteLists of restaurants', 'vteLists of restaurants.1']
9 (2, 2) [0, 1]
10 (1, 2) [0, 1]
11 (3, 2) [0, 1]
12 (20, 2) ['vteMichelin', 'vteMichelin.1']
13 (12, 2) [0, 1]
14 (6,

In [10]:
print(tables[3].head(10))

                Name       Cuisine                    Location 2011  \
0           42 Grams      American            Chicago – Uptown    —   
1             Acadia      American        Chicago – South Loop    —   
2             Alinea      Creative      Chicago – Lincoln Park  NaN   
3            Avenues  Contemporary  Chicago – Magnificent Mile  NaN   
4    Band of Bohemia     Gastropub        Chicago – Ravenswood    —   
5          Blackbird  Contemporary         Chicago – West Loop  NaN   
6               Boka      American      Chicago – Lincoln Park  NaN   
7          Bonsoirée        French      Chicago – Logan Square  NaN   
8  Charlie Trotter's      American      Chicago – Lincoln Park  NaN   
9   Crofton on Wells  Contemporary       Chicago – River North  NaN   

         2012        2013        2014        2015        2016        2017  \
0           —           —           —         NaN         NaN         NaN   
1           —         NaN         NaN         NaN         NaN   

In [11]:
import requests, time, pandas as pd
from io import StringIO
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "UH-student-project/1.0 (your_contact_here)"}
url = "https://en.wikipedia.org/wiki/List_of_Michelin-starred_restaurants_in_Chicago"
resp = requests.get(url, headers=HEADERS, timeout=15)
soup = BeautifulSoup(resp.text, "html.parser")

# 1. see which images the tables contain (alt text, file name, how many)
seen = {}
for img in soup.select("table img"):
    key = (img.get("alt"), (img.get("src") or "").split("/")[-1])
    seen[key] = seen.get(key, 0) + 1
for key, n in sorted(seen.items(), key=lambda x: -x[1])[:15]:
    print(n, key)

# 2. turn the star images into a text symbol
for img in soup.select("table img"):
    info = ((img.get("alt") or "") + " " + (img.get("src") or "")).lower()
    if "star" in info:
        img.replace_with("★")

tables = pd.read_html(StringIO(str(soup)))
for i, t in enumerate(tables):
    print(i, t.shape, list(t.columns)[:6])

big = max(tables, key=len)
print(big.head(12).to_string())

265 ('1 Michelin star', '20px-Etoile_Michelin-1.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail')
53 ('2 Michelin stars', '40px-Etoile_Michelin-2.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail')
24 ('3 Michelin stars', '60px-Etoile_Michelin-3.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail')
3 ('1 Michelin star', '40px-Etoile_Michelin-1.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail')
2 ('1 Michelin green star', '20px-MichelinGreenStar.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail')
2 ('Michelin key', '20px-Clefs_Michelin.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail')
1 ('', '20px-Symbol_category_class.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail')
0 (5, 2) [0, 1]
1 (33, 8) ['Name', 'Cuisine', 'Location', '2021', '2022', '2023']
2 (5, 2) [0, 1]
3 (52, 13) ['Name', 'Cuis

In [12]:
import requests, time, re, pandas as pd
from io import StringIO
from bs4 import BeautifulSoup

BASE = "https://en.wikipedia.org/wiki/List_of_Michelin-starred_restaurants_in_"
HEADERS = {"User-Agent": "UH-student-project/1.0 (mgonzalez131@uh.edu)"}

def scrape_page(page_name):
    resp = requests.get(BASE + page_name, headers=HEADERS, timeout=15)
    time.sleep(1)
    if resp.status_code != 200:
        print("Error", resp.status_code, page_name)
        return pd.DataFrame()

    soup = BeautifulSoup(resp.text, "html.parser")

    # star images -> text (the number of stars is in the alt text)
    for img in soup.select("table img"):
        alt = img.get("alt") or ""
        m = re.match(r"^(\d) Michelin stars?$", alt)
        if m:
            img.replace_with(m.group(1))
        elif "green star" in alt.lower():
            img.replace_with("G")

    tables = pd.read_html(StringIO(str(soup)))
    parts = []
    for t in tables:
        t.columns = [str(c) for c in t.columns]
        year_cols = [c for c in t.columns if re.fullmatch(r"\d{4}", c)]
        if "Name" not in t.columns or not year_cols:
            continue                       # not a restaurant-by-year table
        ids = [c for c in ["Name", "Cuisine", "Location"] if c in t.columns]
        long = t.melt(id_vars=ids, value_vars=year_cols, var_name="year", value_name="raw")
        parts.append(long)

    if not parts:
        return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)

    df["raw"] = df["raw"].astype(str)
    df = df[df["raw"].str.match(r"^[123]")].copy()          # keep only years with stars
    df["stars"] = df["raw"].str[0].astype(int)
    df["green_star"] = df["raw"].str.contains("G")
    df["region"] = page_name.replace("_", " ")
    df = df.rename(columns={"Name": "restaurant", "Cuisine": "cuisine", "Location": "location"})
    return df.drop(columns="raw").drop_duplicates()

# test: one page in detail, then a few more to estimate the size
chicago = scrape_page("Chicago")
print(chicago.shape)
print(chicago.head(10).to_string())
print(chicago["stars"].value_counts())
print(chicago["year"].value_counts().sort_index())

total = len(chicago)
for name in ["Texas", "California", "New_York_City", "France"]:
    d = scrape_page(name)
    print(name, d.shape)
    total += len(d)
print("Rows so far:", total)

(336, 7)
   restaurant       cuisine                location  year  stars  green_star   region
0      Acadia      American    Chicago – South Loop  2021      2       False  Chicago
1      Alinea      Creative  Chicago – Lincoln Park  2021      3       False  Chicago
3        Boka      American  Chicago – Lincoln Park  2021      1       False  Chicago
6    EL Ideas  Contemporary  Chicago – Douglas Park  2021      1       False  Chicago
7   Elizabeth  Contemporary    Chicago – Ravenswood  2021      1       False  Chicago
8       Elske        Nordic     Chicago – West Loop  2021      1       False  Chicago
9     Entente      American   Chicago – River North  2021      1       False  Chicago
11       Ever      Creative     Chicago – West Loop  2021      2       False  Chicago
14  Goosefoot  Contemporary    Chicago – Ravenswood  2021      1       False  Chicago
17       Mako      Japanese     Chicago – West Loop  2021      1       False  Chicago
stars
1    263
2     51
3     22
Name: count,

In [ ]:
import requests, time, re, pandas as pd
from io import StringIO
from bs4 import BeautifulSoup
from tqdm import tqdm

BASE = "https://en.wikipedia.org/wiki/List_of_Michelin-starred_restaurants_in_"
HEADERS = {"User-Agent": "UH-student-project/1.0 (mgonzalez131@uh.edu)"}   # your contact or project URL
OUTPUT_CSV = "michelin_starred_restaurants.csv"
MIN_ROWS = 15000

PAGES = [
    "Abu_Dhabi", "American_Northeast_Cities", "the_American_South", "the_American_Southwest",
    "Argentina", "Austria", "Beijing", "Belgium_&_Luxembourg", "Brazil", "California", "Chengdu",
    "Chicago", "Colorado", "Croatia", "the_Czech_Republic", "Denmark", "Doha", "Dubai", "England",
    "Estonia", "Finland", "Florida", "France", "Germany", "Greece", "Guangzhou", "Hokkaido",
    "Hong_Kong_and_Macau", "Hungary", "Iceland", "Ireland", "Italy", "Japan", "Greater_London",
    "Kyoto_and_Osaka", "Malaysia", "Malta", "Mexico", "Monaco", "Moscow", "Nara", "the_Netherlands",
    "New_York_City", "New_Zealand", "Northern_Ireland", "Norway", "Paris", "the_Philippines",
    "Poland", "Portugal", "Quebec", "Scotland", "Serbia", "Shanghai", "Singapore", "Slovenia",
    "South_Australia", "South_Korea", "Spain", "Sweden", "Switzerland", "Taiwan", "Texas",
    "Thailand", "Tokyo", "Toronto", "Turkey", "Vancouver", "Vietnam", "Wales", "Washington,_D.C.",
]

NAME_ALIASES = ["Name", "Restaurant"]
CUISINE_ALIASES = ["Cuisine", "Cuisine type", "Type", "Style"]
LOCATION_ALIASES = ["Location", "Ward", "City", "District", "Town", "Area", "Neighborhood",
                    "Neighbourhood", "Borough", "Prefecture", "Region", "Location (province)"]

def clean_text(c):
    return re.sub(r"\[.*?\]", "", str(c)).strip()      # removes footnote marks like [a] or [12]

def pick(columns, aliases):
    for a in aliases:
        for c in columns:
            if c.lower() == a.lower():
                return c
    return None

def scrape_page(page_name):
    resp = requests.get(BASE + page_name, headers=HEADERS, timeout=15)
    time.sleep(1)
    if resp.status_code != 200:
        print("Error", resp.status_code, page_name)
        return pd.DataFrame()

    soup = BeautifulSoup(resp.text, "html.parser")

    # star images -> text (the number of stars is in the alt text)
    for img in soup.select("table img"):
        alt = img.get("alt") or ""
        m = re.match(r"^(\d) Michelin stars?$", alt)
        if m:
            img.replace_with(m.group(1))
        elif "green star" in alt.lower():
            img.replace_with("G")

    tables = pd.read_html(StringIO(str(soup)))
    parts = []
    for t in tables:
        t.columns = [clean_text(c) for c in t.columns]
        t = t.loc[:, ~t.columns.duplicated()]

        year_cols = [c for c in t.columns if re.fullmatch(r"\d{4}", c)]
        name_c = pick(t.columns, NAME_ALIASES)
        if name_c is None or not year_cols:
            continue                                    # not a restaurant-by-year table

        cuisine_c = pick(t.columns, CUISINE_ALIASES)
        loc_c = pick(t.columns, LOCATION_ALIASES)
        if cuisine_c is None or loc_c is None:
            print("Check columns in", page_name, "->", list(t.columns)[:6])

        rename = {name_c: "restaurant"}
        if cuisine_c:
            rename[cuisine_c] = "cuisine"
        if loc_c:
            rename[loc_c] = "location"
        t = t.rename(columns=rename)
        parts.append(t.melt(id_vars=list(rename.values()), value_vars=year_cols,
                            var_name="year", value_name="raw"))

    if not parts:
        return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    for c in ["cuisine", "location"]:
        if c not in df.columns:
            df[c] = None

    df["raw"] = df["raw"].astype(str)
    df = df[df["raw"].str.match(r"^[123]")].copy()
    df["stars"] = df["raw"].str[0].astype(int)
    df["green_star"] = df["raw"].str.contains("G")
    df["year"] = df["year"].astype(int)
    df["region"] = page_name.replace("_", " ")

    for c in ["restaurant", "cuisine", "location"]:       # remove footnote marks inside the values
        df[c] = df[c].astype("string").str.replace(r"\[.*?\]", "", regex=True).str.strip()
    return df.drop(columns="raw")

all_parts = []
failed = []
for name in tqdm(PAGES, desc="Scraping Michelin lists"):
    try:
        d = scrape_page(name)
    except Exception as e:
        print("Problem with", name, "->", type(e).__name__)
        failed.append(name)
        continue
    if d.empty:
        print("No data:", name)
    else:
        all_parts.append(d)

df = pd.concat(all_parts, ignore_index=True)
df = df.drop_duplicates(subset=["restaurant", "location", "year", "stars"])   # some pages overlap (e.g. Japan / Tokyo)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved {len(df)} rows and {df.shape[1]} columns to {OUTPUT_CSV}")
print("Pages that failed:", failed)
print("Reached the minimum of", MIN_ROWS, "rows:", len(df) >= MIN_ROWS)

Scraping Michelin lists:  15%|█▌        | 11/71 [00:17<01:34,  1.58s/it]

Check columns in Chengdu -> ['Name', 'Cuisine', '2022', '2023', '2024', '2025']


Scraping Michelin lists:  39%|███▉      | 28/71 [00:44<01:16,  1.79s/it]

Check columns in Hong_Kong_and_Macau -> ['Name', 'Territory', 'Location', '2009', '2010', '2011']


Scraping Michelin lists:  44%|████▎     | 31/71 [00:49<01:05,  1.63s/it]

Check columns in Ireland -> ['Name', 'Location', '2010', '2011', '2012', '2013']
Check columns in Ireland -> ['Name', 'Location', '2000', '2001', '2002', '2003']
Check columns in Ireland -> ['Name', 'Location', '1990', '1991', '1992', '1993']
Check columns in Ireland -> ['Name', 'Location', '1980', '1981', '1982', '1983']
Check columns in Ireland -> ['Name', 'Location', '1974', '1975', '1976', '1977']


... (6 outputs left out)

Scraping Michelin lists:  90%|█████████ | 64/71 [01:43<00:10,  1.52s/it]

Check columns in Thailand -> ['Restaurant', 'Cuisine', 'Location (province)', '2018', '2019', '2020']


Scraping Michelin lists: 100%|██████████| 71/71 [01:53<00:00,  1.60s/it]


Saved 24166 rows and 7 columns to michelin_starred_restaurants.csv
Pages that failed: []
Reached the minimum of 15000 rows: True


In [16]:
check = df.groupby("region")[["cuisine", "location"]].agg(lambda s: s.isna().sum())
print(check[(check["cuisine"] > 0) | (check["location"] > 0)])

                      cuisine  location
region                                 
Beijing                     3         0
Belgium & Luxembourg        1        13
Chengdu                     0        74
Hong Kong and Macau       842         4
Ireland                   206         0
Kyoto and Osaka             1         0
Moscow                      0         9
Northern Ireland            8         0
Portugal                   27         0
Slovenia                    3         0
Switzerland                49         0
Thailand                    0       281
Wales                      26         0
the Netherlands          3301         0
